### Validation  of dimension


In [2]:
import os
os.chdir("..")
from src.ingestion.database import get_engine
engine = get_engine()

In [3]:
import pandas as pd

dimension_tables = [
    "dim_customer",
    "dim_product",
    "dim_seller",
    "dim_geography",
    "dim_date"
]

dimension_counts = []

for table in dimension_tables:

    query = f"SELECT COUNT(*) AS row_count FROM {table}"

    count = pd.read_sql(query, engine)["row_count"].iloc[0]

    dimension_counts.append({
        "table": table,
        "row_count": count
    })

dimension_counts_df = pd.DataFrame(dimension_counts)

dimension_counts_df

,table,row_count
0,dim_customer,99441
1,dim_product,32951
2,dim_seller,3095
3,dim_geography,19015
4,dim_date,634


In [4]:
result = pd.read_sql("""
SELECT COUNT(*) AS errors
FROM fact_sales
WHERE total_item_value <> price + freight_value;
""", engine)

result

,errors
0,0


In [6]:
pd.read_sql("""
SELECT
    order_id,
    delivery_days,
    delivery_delay_days,
    is_delayed
FROM fact_sales
WHERE delivered_date IS NOT NULL
LIMIT 5;
""", engine)

,order_id,delivery_days,delivery_delay_days,is_delayed
0,7f39ba4c9052be115350065d07583cac,9,-13,0
1,9dc8d1a6f16f1b89874c29c9d8d30447,12,-13,0
2,d455a8cb295653b55abda06d434ab492,11,-23,0
3,006e43460a55bc60c0a437521e426529,8,-14,0
4,00dfb074b5c910fbd08e04691c4b712f,7,-25,0


In [7]:
query = """
SELECT
    SUM(customer_key IS NULL) AS missing_customers,
    SUM(product_key IS NULL) AS missing_products,
    SUM(seller_key IS NULL) AS missing_sellers,
    SUM(purchase_date_key IS NULL) AS missing_dates,
    SUM(customer_geography_key IS NULL) AS missing_customer_geo,
    SUM(seller_geography_key IS NULL) AS missing_seller_geo
FROM fact_sales;
"""

key_check = pd.read_sql(query, engine)

key_check

,missing_customers,missing_products,missing_sellers,missing_dates,missing_customer_geo,missing_seller_geo
0,0.0,0.0,0.0,0.0,302.0,253.0


## Revenue summary

In [8]:
query = """
SELECT
    COUNT(DISTINCT order_id) AS total_orders,

    COUNT(*) AS total_items,

    ROUND(SUM(price), 2) AS product_revenue,

    ROUND(SUM(freight_value), 2) AS freight_revenue,

    ROUND(SUM(total_item_value), 2) AS total_revenue,

    ROUND(
        SUM(total_item_value)
        / COUNT(DISTINCT order_id),
        2
    ) AS average_order_value,

    ROUND(
        SUM(total_item_value)
        / COUNT(*),
        2
    ) AS average_item_value

FROM fact_sales;
"""

sales_summary = pd.read_sql(query, engine)

sales_summary

,total_orders,total_items,product_revenue,freight_revenue,total_revenue,average_order_value,average_item_value
0,98666,112650,13591643.7,2251909.54,15843553.24,160.58,140.64
